# Inverse covariance estimation with MFCF-LoGo.

Sanity-check MFCFLoGo against GraphicalLassoCV on synthetic data whose
true precision is *not* chordal — in both the well-posed regime (n > p)
and the high-dimensional regime (n << p) MFCF is designed for.

Each scenario also saves a side-by-side heatmap of the ground-truth
precision and every estimator's reconstruction to ``./figs/``.

Scenarios
---------
* ``scenario_er``                  Erdős-Rényi sparse precision, Gaussian.
* ``scenario_cycle``               Length-p cycle + a few chords, Gaussian.
* ``scenario_blocks_gaussian``     Block-diagonal precision, Gaussian.
* ``scenario_blocks_nongaussian``  Non-Gaussian latent-block features (cos/
                                   sin/quadratic of a shared latent).
* ``scenario_mi_collapse``         Sample-rich Gaussian — Linfoot-KSG MI
                                   collapses to ``|Pearson rho|`` and the
                                   two MFCF paths become identical.
The first four scenarios run in two regimes: ``n > p`` and ``n << p``.

Design rationale
----------------
The three Gaussian generators form a deliberate triangle of difficulty,
chosen so that no single graph property silently advantages MFCF:

* **ER** is the *neutral* baseline — generic random sparse precision
  with no exploitable structure, almost-surely non-chordal once
  ``p * edge_prob >= 2``. It is also the standard benchmark in the
  GLasso / neighbourhood-selection literature, so it tests MFCF on
  its competitor's home turf. The single ``edge_prob`` knob lets the
  same generator probe both the ``n > p`` and ``n << p`` regimes.
* **Cycle + chords** is the *adversarial* target: a length-``p`` cycle
  is the canonical non-chordal graph for ``p >= 4``. MFCF can only
  approximate it via a chordal cover, so this scenario quantifies the
  unavoidable approximation cost — it is the worst case for any
  chordal-by-construction estimator.
* **Block-diagonal** is the *friendly* target: chordal within each
  block, hard conditional independence across blocks. MFCF is
  expected to match or beat GLasso here.

``scenario_blocks_nongaussian`` is the only place where MFCF-MI is
*supposed* to beat MFCF-corr: the latent block-mates are linked by
non-monotone maps (``cos(2z)``, ``sin(2z)``, ``z^2 - 1``, ``|z| - 0.8``),
which drive Pearson correlation to near zero while leaving the
mutual information large.

``scenario_mi_collapse`` is the dual control: a sample-rich Gaussian
where, by Linfoot's 1957 identity, normalised KSG-MI must converge
to ``|Pearson rho|``. With enough samples the two MFCF paths
*must* land on the same edges; the scenario quantifies how tight
that collapse really is at finite n.

In [1]:
import os
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
from optuna.samplers import TPESampler
from scipy import linalg
from sklearn.covariance import GraphicalLassoCV, empirical_covariance, log_likelihood
from sklearn.exceptions import ConvergenceWarning

from mfcf_logo import MFCFLoGo
from mutual_information import mutual_information_matrix

### Define global hyperparameters

In [2]:
optuna.logging.set_verbosity(optuna.logging.ERROR)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

HPO_TIME_BUDGET     = 120.0   # seconds, per (scenario, similarity)
HPO_N_SUBSAMPLES    = 8      # subsample-without-replacement count per trial
HPO_SUBSAMPLE_FRAC  = 0.70   # fraction of rows in each subsample (no replacement)
HPO_INSTAB_TARGET   = 0.10   # StARS-style instability target; excess is penalised
HPO_INSTAB_PENALTY  = 5.0    # hinge weight on (instability - target), units: score
HPO_DENSITY_GAIN    = 5.0    # max possible density-reward (asymptote of tanh)
HPO_DENSITY_SCALE   = 0.03   # density at which tanh reaches ~0.76 of its asymptote
HPO_LL_TIEBREAK     = 0.01   # small weight on OOB log-lik (final-stage tiebreaker)
HPO_TPE_STARTUP     = 12     # random trials before TPE surrogate kicks in
HPO_SEED            = 42

### Define the data generators

In [3]:
def generate_er_precision(p, edge_prob, prng,
                          weight_scale=0.3, diag_margin=0.5):
    """Erdős-Rényi off-diagonal pattern; diagonally dominant for SPD.

    Role
    ----
    Neutral baseline. No block / chordal / cycle structure is imposed,
    so the resulting graph is almost-surely non-chordal once
    ``p * edge_prob >= 2``. This is the canonical sparse-precision
    benchmark from the GLasso literature — putting it here means GLasso
    is being compared on its home turf. Diagonal dominance
    (``|row sum| + margin``) is the cheap way to guarantee SPD without
    constraining the off-diagonal pattern.
    """
    mask = np.triu(prng.uniform(size=(p, p)) < edge_prob, k=1)
    P = np.zeros((p, p))
    P[mask] = prng.uniform(-weight_scale, weight_scale, size=int(mask.sum()))
    P = P + P.T
    np.fill_diagonal(P, np.abs(P).sum(axis=1) + diag_margin)
    return P


def generate_cycle_plus_chords_precision(p, n_chords, prng,
                                         weight_scale=0.3, diag_margin=0.5):
    """Length-p cycle plus a few extra chords — non-chordal for p >= 4.

    Role
    ----
    Adversarial target for any chordal-by-construction estimator.
    A pure cycle of length ``p >= 4`` is the textbook non-chordal
    graph: every chordal cover MFCF can emit must either add fill-in
    edges or drop true ones. The handful of extra chords keeps the
    structure non-trivial without making it chordal. This scenario
    measures the *unavoidable* approximation cost of forcing the
    estimator into a clique forest.
    """
    P = np.zeros((p, p))
    for i in range(p):
        j = (i + 1) % p
        w = prng.uniform(-weight_scale, weight_scale)
        P[i, j] = w
        P[j, i] = w
    placed = 0
    while placed < n_chords:
        i, j = prng.randint(0, p, size=2)
        if i == j or P[i, j] != 0 or abs(i - j) <= 1 or abs(i - j) == p - 1:
            continue
        w = prng.uniform(-weight_scale, weight_scale)
        P[i, j] = w
        P[j, i] = w
        placed += 1
    np.fill_diagonal(P, np.abs(P).sum(axis=1) + diag_margin)
    return P


def generate_block_precision(n_blocks, block_size, prng,
                             weight_scale=0.3, diag_margin=0.5):
    """Block-diagonal sparse precision: every pair of variables in distinct
    blocks is conditionally independent (hard zero in the precision).
    Within each block, edge weights are i.i.d. uniform.

    Role
    ----
    Friendly target. Each block is a complete sub-graph (chordal),
    blocks are conditionally independent, and the union is a forest
    of cliques by construction — exactly the family MFCF is built to
    recover. If MFCF does not match GLasso here, something is wrong.
    """
    p = n_blocks * block_size
    P = np.zeros((p, p))
    tri_mask = np.triu(np.ones((block_size, block_size), dtype=bool), k=1)
    n_block_edges = int(tri_mask.sum())
    for k in range(n_blocks):
        i0 = k * block_size
        i1 = i0 + block_size
        W = np.zeros((block_size, block_size))
        W[tri_mask] = prng.uniform(-weight_scale, weight_scale,
                                   size=n_block_edges)
        W = W + W.T
        P[i0:i1, i0:i1] = W
    np.fill_diagonal(P, np.abs(P).sum(axis=1) + diag_margin)
    return P


def _generate_nonmonotone_blocks(n, n_latents, block_size, seed):
    """Latent-block non-Gaussian features — MI should beat correlation here.

    Role
    ----
    The only scenario where MFCF-MI is *expected* to dominate
    MFCF-corr. Block-mates share a latent ``Z`` but are observed
    through non-monotone (and even-symmetric) maps such as
    ``cos(2z)`` and ``z^2 - 1``. Pearson correlation between two
    such columns is near zero — symmetric noise around the same
    centre — while mutual information remains large because the
    columns are deterministic functions of the same source. This
    is the case-study justifying the MI gain path at all.
    """
    prng = np.random.RandomState(seed)
    p = n_latents * block_size
    Z = prng.randn(n, n_latents)
    fs = [
        lambda z: np.cos(2 * z),
        lambda z: np.sin(2 * z),
        lambda z: z ** 2 - 1.0,
        lambda z: np.abs(z) - 0.8,
    ]
    X = np.empty((n, p))
    true_block = np.empty(p, dtype=int)
    for k in range(n_latents):
        for b in range(block_size):
            fn = fs[b % len(fs)]
            X[:, k * block_size + b] = fn(Z[:, k]) + 0.05 * prng.randn(n)
            true_block[k * block_size + b] = k
    prec_true = (true_block[:, None] == true_block[None, :]).astype(float)
    return X, prec_true


def _standardise(prec):
    """Move (cov, prec) to the unit-diagonal-covariance parametrisation.

    Sampling and edge-set comparisons are invariant under per-feature
    rescaling, so we always evaluate in the canonical form where
    ``diag(Sigma) == 1``. Without this, large random diagonal entries
    in the precision would dominate the per-panel colour scale and
    visually wash out the off-diagonal pattern we actually care about.
    """
    cov = linalg.inv(prec)
    d = np.sqrt(np.diag(cov))
    cov = cov / d / d[:, None]
    prec = prec * d * d[:, None]
    return cov, prec

### Define the scoring functions

In [4]:
def _edges_from_precision(P, thr=1e-8):
    A = np.abs(P) > thr
    np.fill_diagonal(A, False)
    p = P.shape[0]
    return {(i, j) for i in range(p) for j in range(i + 1, p) if A[i, j]}


def _edge_scores(E_est, E_true):
    tp = len(E_est & E_true)
    fp = len(E_est - E_true)
    fn = len(E_true - E_est)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return prec, rec, f1


def _evaluate(name, model, X, E_true):
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        warnings.simplefilter("ignore", RuntimeWarning)
        model.fit(X)
    dt = time.time() - t0
    p, r, f = _edge_scores(_edges_from_precision(model.precision_), E_true)
    print(f"  {name:28s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  ({dt:.2f}s)")
    return model.precision_.copy()

### Hyperparameter tuning

In [5]:
def _mfcf_hpo(
    X: np.ndarray,
    similarity: str,
    *,
    time_budget:        float = HPO_TIME_BUDGET,
    n_subsamples:       int   = HPO_N_SUBSAMPLES,
    subsample_frac:     float = HPO_SUBSAMPLE_FRAC,
    instability_target: float = HPO_INSTAB_TARGET,
    instability_penalty: float = HPO_INSTAB_PENALTY,
    density_gain:       float = HPO_DENSITY_GAIN,
    density_scale:      float = HPO_DENSITY_SCALE,
    ll_tiebreak_weight: float = HPO_LL_TIEBREAK,
    n_startup_trials:   int   = HPO_TPE_STARTUP,
    seed:               int   = HPO_SEED,
) -> tuple[MFCFLoGo, dict, int, dict]:
    """Optuna TPE + StARS-style HPO for :class:`MFCFLoGo`, leakage-free.

    See the design block above this function for the objective and
    sampler rationale.

    Returns
    -------
    final_model : MFCFLoGo
        Refit on the full ``X`` with the best parameters found.
    best_params : dict
        The winning hyper-parameter dictionary.
    n_trials : int
        Number of Optuna trials that fit inside the time budget.
    diagnostics : dict
        ``{"best_value", "best_density", "best_instability",
        "best_oob_ll"}`` — the components of the winning score, for
        post-hoc inspection.
    """
    n_samples, n_features = X.shape
    max_cs_upper = max(2, n_features - 1)
    sub_size = max(int(subsample_frac * n_samples), 5)
    sub_size = min(sub_size, n_samples - 5)   # guarantee non-empty OOB

    rng_master = np.random.RandomState(seed)
    subsamples: list[tuple[np.ndarray, np.ndarray]] = []
    all_idx = np.arange(n_samples)
    for _ in range(n_subsamples):
        tr = rng_master.choice(n_samples, size=sub_size, replace=False)
        oob = np.setdiff1d(all_idx, tr)
        if oob.size >= 5:
            subsamples.append((tr, oob))
    if not subsamples:
        raise RuntimeError(
            "Could not draw any subsample with a non-trivial OOB. "
            "Reduce subsample_frac or increase n_samples."
        )

    iu = np.triu_indices(n_features, k=1)

    def objective(trial: optuna.Trial) -> float:
        threshold = trial.suggest_float("threshold", 0.0, 0.8)
        # Static ranges for both clique-size knobs; ``min <= max``
        # is enforced by post-sample clamping. TPE handles dynamic
        # ranges natively, but static + clamp keeps the parameter
        # marginals comparable across trials.
        max_cs_raw = trial.suggest_int("max_clique_size", 2, max_cs_upper)
        min_cs_raw = trial.suggest_int("min_clique_size", 1, 5)
        min_cs = min(min_cs_raw, max_cs_raw)
        max_cs = max(min_cs_raw, max_cs_raw)
        coord_num = trial.suggest_int(
            "coordination_number", 1, max(2, n_features), log=True,
        )
        if similarity == "mutual_information":
            mi_k    = trial.suggest_int("mi_n_neighbors", 2, 20)
            mi_norm = trial.suggest_categorical("mi_normalize",
                                                ["linfoot", "none"])
        else:
            mi_k, mi_norm = 3, "linfoot"

        edge_indicators: list[np.ndarray] = []
        oob_lls:         list[float]      = []
        for tr_idx, oob_idx in subsamples:
            est = MFCFLoGo(
                threshold=threshold,
                min_clique_size=min_cs,
                max_clique_size=max_cs,
                coordination_number=coord_num,
                similarity=similarity,
                mi_n_neighbors=mi_k,
                mi_normalize=mi_norm,
                mi_random_state=seed,
            )
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    est.fit(X[tr_idx])
            except Exception:
                # One failed subsample invalidates the trial — a
                # partial mean would poison TPE's good/bad split
                # with a noisy outlier.
                return float("-inf")
            if not np.all(np.isfinite(est.precision_)):
                return float("-inf")
            P_b = est.precision_
            E_b = (np.abs(P_b) > 1e-8).astype(np.int8)
            np.fill_diagonal(E_b, 0)
            edge_indicators.append(E_b)
            S_oob = empirical_covariance(X[oob_idx], assume_centered=False)
            oob_lls.append(float(log_likelihood(S_oob, P_b)))

        E_stack = np.stack(edge_indicators, axis=0)        # (B, p, p)
        p_edge  = E_stack.mean(axis=0)                     # (p, p)
        instab  = 2.0 * p_edge * (1.0 - p_edge)            # in [0, 0.5]
        mean_density     = float(p_edge[iu].mean())
        mean_instability = float(instab[iu].mean())
        mean_oob_ll      = float(np.mean(oob_lls))

        trial.set_user_attr("density",     mean_density)
        trial.set_user_attr("instability", mean_instability)
        trial.set_user_attr("oob_ll",      mean_oob_ll)

        # Primary signal: saturating density reward (steep at zero,
        # caps near ``density_scale`` so the optimiser is not pushed
        # toward dense false-positive graphs once the recovery
        # target's natural density is reached).
        # Hinge penalty discourages high-instability dense solutions.
        # Log-lik is a soft tiebreaker among similar trials.
        excess  = max(0.0, mean_instability - instability_target)
        score   = (density_gain * float(np.tanh(mean_density / density_scale))
                   - instability_penalty * excess
                   + ll_tiebreak_weight  * mean_oob_ll)
        return score

    sampler = TPESampler(
        seed=seed,
        multivariate=True,
        group=True,
        n_startup_trials=n_startup_trials,
    )
    study = optuna.create_study(direction="maximize", sampler=sampler)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        study.optimize(
            objective,
            timeout=time_budget,
            n_jobs=1,
            show_progress_bar=False,
        )

    feasible = [t for t in study.trials if np.isfinite(t.value or float("-inf"))]
    if not feasible:
        warnings.warn(
            f"HPO ({similarity}) found no feasible trial; "
            f"refitting with MFCFLoGo defaults.",
            RuntimeWarning,
        )
        best = dict(
            threshold=0.0,
            min_clique_size=1,
            max_clique_size=min(4, max_cs_upper),
            coordination_number=int(max(2, n_features)),
        )
        diagnostics = dict(best_value=float("-inf"), best_density=0.0,
                           best_instability=0.0, best_oob_ll=float("-inf"))
    else:
        best_trial = max(feasible, key=lambda t: t.value)
        best = dict(best_trial.params)
        diagnostics = dict(
            best_value=float(best_trial.value),
            best_density=float(best_trial.user_attrs.get("density", 0.0)),
            best_instability=float(best_trial.user_attrs.get("instability", 0.0)),
            best_oob_ll=float(best_trial.user_attrs.get("oob_ll", float("-inf"))),
        )

    # Apply the same clamp the objective applied.
    min_cs_raw = best.get("min_clique_size", 1)
    max_cs_raw = best.get("max_clique_size", min(4, max_cs_upper))
    best_min_cs = min(min_cs_raw, max_cs_raw)
    best_max_cs = max(min_cs_raw, max_cs_raw)
    final = MFCFLoGo(
        threshold=best.get("threshold", 0.0),
        min_clique_size=best_min_cs,
        max_clique_size=best_max_cs,
        coordination_number=best.get("coordination_number",
                                     int(max(2, n_features))),
        similarity=similarity,
        mi_n_neighbors=best.get("mi_n_neighbors", 3),
        mi_normalize=best.get("mi_normalize", "linfoot"),
        mi_random_state=seed,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        final.fit(X)
    return final, best, len(study.trials), diagnostics


def _evaluate_mfcf_hpo(name, X, similarity, E_true,
                       time_budget=HPO_TIME_BUDGET):
    """Run the leakage-free HPO and report edge-set scores.

    The (single) point at which HPO meets the evaluation target is
    *outside* the HPO loop: only the refit precision is scored
    against ``E_true``. Optuna never sees ``E_true``.
    """
    t0 = time.time()
    final, best, n_trials, diag = _mfcf_hpo(
        X, similarity=similarity, time_budget=time_budget,
    )
    dt = time.time() - t0
    P = final.precision_
    p, r, f = _edge_scores(_edges_from_precision(P), E_true)
    cs_lo = min(best.get("min_clique_size", 1), best.get("max_clique_size", 1))
    cs_hi = max(best.get("min_clique_size", 1), best.get("max_clique_size", 1))
    print(f"  {name:28s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  "
          f"({dt:.1f}s, {n_trials} trials, "
          f"clique=[{cs_lo},{cs_hi}], thr={best.get('threshold', 0.0):.3f}, "
          f"dens={diag['best_density']:.3f}, instab={diag['best_instability']:.3f})")
    return P.copy()

### Define the plotting functions

In [6]:
def _zero_diag(P):
    P = P.copy()
    np.fill_diagonal(P, 0.0)
    return P


def _panel_vmax(M):
    """99th-percentile of non-zero |entries|, with safe fallback."""
    absM = np.abs(M)
    nz = absM[absM > 0]
    if nz.size == 0:
        return 1.0
    v = float(np.quantile(nz, 0.99))
    return v if np.isfinite(v) and v > 0 else float(absM.max() or 1.0)


def _save_comparison(P_true, panels, outfile, suptitle):
    """``panels`` is a list of ``(title, precision_matrix)``.

    The diagonal is masked out so the visualisation focuses on the
    conditional-dependence structure rather than the unit diagonal.
    Each panel uses its own colour scale (clipped to the 99th percentile
    of non-zero entries) because MFCF and GLasso precisions can live at
    wildly different magnitudes — a shared scale washes one out.
    """
    matrices = [_zero_diag(P_true)] + [_zero_diag(P) for _, P in panels]
    titles = ["Ground truth"] + [t for t, _ in panels]
    n = len(matrices)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.8))
    for ax, title, M in zip(axes, titles, matrices):
        vmax = _panel_vmax(M)
        im = ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                       interpolation="nearest")
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, shrink=0.75, fraction=0.046, pad=0.02)
    fig.suptitle(suptitle, fontsize=11)
    fig.savefig(outfile, dpi=120, bbox_inches="tight")
    plt.close(fig)

### Define the well-posed $(n > p)$ scenarios

In [7]:
def scenario_er(seed=1):
    p, n = 60, 400
    prng = np.random.RandomState(seed)
    prec = generate_er_precision(p, edge_prob=0.06, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[ER precision, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "er_lowdim.png"),
        f"Erdős-Rényi precision  (p={p}, n={n})",
    )
    '''


def scenario_cycle(seed=1):
    p, n = 40, 300
    prng = np.random.RandomState(seed)
    prec = generate_cycle_plus_chords_precision(p, n_chords=5, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Cycle + chords, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "cycle_lowdim.png"),
        f"Cycle + chords  (p={p}, n={n})",
    )
    '''


def scenario_blocks_gaussian(seed=1):
    n_blocks, block_size, n = 5, 4, 400
    prng = np.random.RandomState(seed)
    prec = generate_block_precision(n_blocks, block_size, prng)
    cov, prec = _standardise(prec)
    p = n_blocks * block_size
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Block-diag, Gaussian, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_gaussian_lowdim.png"),
        f"Block-diag precision, Gaussian  (p={p}, n={n})",
    )
    '''


def scenario_blocks_nongaussian(seed=1):
    n_latents, block_size, n = 5, 4, 600
    X, prec_true = _generate_nonmonotone_blocks(n, n_latents, block_size, seed)
    E_true = _edges_from_precision(prec_true)
    p = n_latents * block_size
    print(f"\n[Non-Gaussian blocks, n>p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_corr = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec_true,
        [("MFCF corr HPO", P_corr), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_nongaussian_lowdim.png"),
        f"Non-Gaussian blocks  (p={p}, n={n})",
    )
    '''

### Define the high-dimensional scenarios $(n << p)$

In [8]:
def scenario_er_highdim(seed=1):
    p, n = 150, 50
    prng = np.random.RandomState(seed)
    prec = generate_er_precision(p, edge_prob=0.025, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[ER precision, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "er_highdim.png"),
        f"Erdős-Rényi precision  (p={p}, n={n}, n<<p)",
    )
    '''


def scenario_cycle_highdim(seed=1):
    p, n = 80, 30
    prng = np.random.RandomState(seed)
    prec = generate_cycle_plus_chords_precision(p, n_chords=8, prng=prng)
    cov, prec = _standardise(prec)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Cycle + chords, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "cycle_highdim.png"),
        f"Cycle + chords  (p={p}, n={n}, n<<p)",
    )
    '''


def scenario_blocks_gaussian_highdim(seed=1):
    n_blocks, block_size, n = 6, 8, 40
    prng = np.random.RandomState(seed)
    prec = generate_block_precision(n_blocks, block_size, prng)
    cov, prec = _standardise(prec)
    p = n_blocks * block_size
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[Block-diag, Gaussian, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_mfcf = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_mfcf), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_gaussian_highdim.png"),
        f"Block-diag precision, Gaussian  (p={p}, n={n}, n<<p)",
    )
    '''


def scenario_blocks_nongaussian_highdim(seed=1):
    n_latents, block_size, n = 6, 8, 40
    X, prec_true = _generate_nonmonotone_blocks(n, n_latents, block_size, seed)
    E_true = _edges_from_precision(prec_true)
    p = n_latents * block_size
    print(f"\n[Non-Gaussian blocks, n<<p]  p={p}, n={n}, |E_true|={len(E_true)}")
    P_corr = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    P_gl   = _evaluate("GraphicalLassoCV",
                       GraphicalLassoCV(n_jobs=-1), X, E_true)
    '''
    _save_comparison(
        prec_true,
        [("MFCF corr HPO", P_corr), ("MFCF MI HPO", P_mi), ("GLasso CV", P_gl)],
        os.path.join(FIG_DIR, "blocks_nongaussian_highdim.png"),
        f"Non-Gaussian blocks  (p={p}, n={n}, n<<p)",
    )
    '''

### Define the "collapse" scenario

In [9]:
def scenario_mi_collapse(seed=1):
    """Linfoot-normalized KSG MI equals ``|Pearson rho|`` under Gaussianity
    (Linfoot 1957).  With enough samples the KSG estimator converges to
    that asymptote and MFCF-MI becomes indistinguishable from MFCF-corr.

    This scenario quantifies the collapse: it reports the elementwise
    distance between the two similarity matrices and shows the MFCF
    precisions land at the same edges.
    """
    # Construct cov directly: 3 equi-correlated blocks of size 4 with
    # within-block rho=0.7.  This gives sizeable signal correlations,
    # so the KSG bias does not dominate at finite n.
    n_blocks, block_size, n = 3, 4, 10000
    p = n_blocks * block_size
    rho = 0.7
    cov = np.eye(p)
    for k in range(n_blocks):
        i0 = k * block_size
        i1 = i0 + block_size
        cov[i0:i1, i0:i1] = (1.0 - rho) * np.eye(block_size) + rho
    prec = linalg.inv(cov)
    prng = np.random.RandomState(seed)
    X = prng.multivariate_normal(np.zeros(p), cov, size=n)
    E_true = _edges_from_precision(prec)
    print(f"\n[MI collapse to |Pearson|]  p={p}, n={n}, |E_true|={len(E_true)}")

    C_corr = np.abs(np.corrcoef(X, rowvar=False))
    C_mi = mutual_information_matrix(X, n_neighbors=3, normalize="linfoot")
    np.fill_diagonal(C_corr, 0.0)
    np.fill_diagonal(C_mi, 0.0)
    diff = np.abs(C_corr - C_mi)
    rho_sim = float(np.corrcoef(C_corr.ravel(), C_mi.ravel())[0, 1])
    print(f"  similarity-matrix collapse:")
    print(f"    max |Linfoot-MI - |corr||         = {diff.max():.4f}")
    print(f"    mean|Linfoot-MI - |corr||         = {diff.mean():.4f}")
    print(f"    Pearson(MI, |corr|) entrywise     = {rho_sim:.4f}")

    P_corr = _evaluate_mfcf_hpo("MFCF corr  HPO", X, "correlation",        E_true)
    P_mi   = _evaluate_mfcf_hpo("MFCF MI    HPO", X, "mutual_information", E_true)
    E_corr = _edges_from_precision(P_corr)
    E_mi = _edges_from_precision(P_mi)
    print(f"  precision-output collapse:")
    print(f"    Jaccard(E_corr, E_mi)             = "
          f"{len(E_corr & E_mi) / max(len(E_corr | E_mi), 1):.4f}")
    print(f"    max |P_corr - P_mi|               = "
          f"{np.abs(P_corr - P_mi).max():.4f}")

    '''
    _save_comparison(
        prec,
        [("MFCF corr HPO", P_corr), ("MFCF MI HPO", P_mi)],
        os.path.join(FIG_DIR, "mi_collapse.png"),
        f"MI collapse to |Pearson|, Gaussian  (p={p}, n={n})",
    )
    '''

### Running tests

In [10]:
scenario_er()


[ER precision, n>p]  p=60, n=400, |E_true|=114
  MFCF corr  HPO                P=0.315  R=0.465  F1=0.376  (120.2s, 2894 trials, clique=[2,4], thr=0.000, dens=0.096, instab=0.099)
  MFCF MI    HPO                P=0.234  R=0.263  F1=0.248  (120.2s, 241 trials, clique=[3,42], thr=0.020, dens=0.076, instab=0.101)
  GraphicalLassoCV              P=0.349  R=0.702  F1=0.466  (4.04s)


In [11]:
scenario_cycle()


[Cycle + chords, n>p]  p=40, n=300, |E_true|=45
  MFCF corr  HPO                P=0.484  R=0.333  F1=0.395  (120.3s, 4263 trials, clique=[3,34], thr=0.032, dens=0.094, instab=0.085)
  MFCF MI    HPO                P=0.239  R=0.378  F1=0.293  (120.3s, 594 trials, clique=[2,36], thr=0.020, dens=0.084, instab=0.099)
  GraphicalLassoCV              P=0.285  R=0.867  F1=0.429  (1.24s)


In [12]:
scenario_blocks_gaussian()


[Block-diag, Gaussian, n>p]  p=20, n=400, |E_true|=30
  MFCF corr  HPO                P=0.697  R=0.767  F1=0.730  (120.4s, 5638 trials, clique=[2,3], thr=0.004, dens=0.178, instab=0.100)
  MFCF MI    HPO                P=0.545  R=0.400  F1=0.462  (120.2s, 1751 trials, clique=[1,3], thr=0.051, dens=0.109, instab=0.094)
  GraphicalLassoCV              P=0.397  R=0.900  F1=0.551  (0.10s)


In [13]:
scenario_blocks_nongaussian()


[Non-Gaussian blocks, n>p]  p=20, n=600, |E_true|=30
  MFCF corr  HPO                P=1.000  R=0.433  F1=0.605  (120.4s, 5778 trials, clique=[1,19], thr=0.317, dens=0.065, instab=0.011)
  MFCF MI    HPO                P=0.800  R=0.400  F1=0.533  (120.2s, 1162 trials, clique=[3,19], thr=0.684, dens=0.079, instab=0.075)
  GraphicalLassoCV              P=0.185  R=0.967  F1=0.310  (2.19s)


In [14]:
scenario_er_highdim()


[ER precision, n<<p]  p=150, n=50, |E_true|=287
  MFCF corr  HPO                P=0.061  R=0.132  F1=0.084  (120.1s, 778 trials, clique=[5,127], thr=0.020, dens=0.066, instab=0.095)
  MFCF MI    HPO                P=0.038  R=0.094  F1=0.054  (120.3s, 472 trials, clique=[5,46], thr=0.014, dens=0.062, instab=0.096)
  GraphicalLassoCV              P=0.429  R=0.021  F1=0.040  (2.69s)


In [15]:
scenario_cycle_highdim()


[Cycle + chords, n<<p]  p=80, n=30, |E_true|=88
  MFCF corr  HPO                P=0.036  R=0.341  F1=0.065  (120.2s, 2048 trials, clique=[2,58], thr=0.001, dens=0.296, instab=0.330)
  MFCF MI    HPO                P=0.046  R=0.125  F1=0.068  (120.1s, 1631 trials, clique=[1,23], thr=0.080, dens=0.068, instab=0.098)
  GraphicalLassoCV              P=0.167  R=0.023  F1=0.040  (1.61s)


In [16]:
scenario_blocks_gaussian_highdim()


[Block-diag, Gaussian, n<<p]  p=48, n=40, |E_true|=168
  MFCF corr  HPO                P=0.147  R=0.613  F1=0.237  (120.2s, 3338 trials, clique=[1,35], thr=0.000, dens=0.639, instab=0.381)
  MFCF MI    HPO                P=0.161  R=0.089  F1=0.115  (120.2s, 2211 trials, clique=[3,21], thr=0.050, dens=0.071, instab=0.094)
  GraphicalLassoCV              P=0.200  R=0.012  F1=0.022  (0.30s)


In [17]:
scenario_blocks_nongaussian_highdim()


[Non-Gaussian blocks, n<<p]  p=48, n=40, |E_true|=168
  MFCF corr  HPO                P=0.645  R=0.357  F1=0.460  (120.2s, 2711 trials, clique=[2,3], thr=0.013, dens=0.082, instab=0.047)
  MFCF MI    HPO                P=0.783  R=0.643  F1=0.706  (120.3s, 2687 trials, clique=[4,4], thr=0.621, dens=0.112, instab=0.080)
  GraphicalLassoCV              P=0.427  R=0.524  F1=0.471  (3.06s)


In [18]:
scenario_mi_collapse()


[MI collapse to |Pearson|]  p=12, n=10000, |E_true|=18
  similarity-matrix collapse:
    max |Linfoot-MI - |corr||         = 0.1804
    mean|Linfoot-MI - |corr||         = 0.0318
    Pearson(MI, |corr|) entrywise     = 0.9857
  MFCF corr  HPO                P=1.000  R=1.000  F1=1.000  (120.4s, 6126 trials, clique=[1,9], thr=0.365, dens=0.273, instab=0.000)
  MFCF MI    HPO                P=1.000  R=1.000  F1=1.000  (121.1s, 98 trials, clique=[1,5], thr=0.059, dens=0.273, instab=0.000)
  precision-output collapse:
    Jaccard(E_corr, E_mi)             = 1.0000
    max |P_corr - P_mi|               = 0.0563
